In [1]:
import json
import pandas as pd
from pathlib import Path

In [2]:
base = Path().cwd()

data_path = base / "data"
results_path = base / "results"
chat_file = "sensor-actuator_single_gpt-4o-mini_n-2_acc-0.511_04.12.2025-10:37:05.json"
chat_file = results_path / chat_file

chat = json.loads(chat_file.read_text())

responses = chat["responses"]

In [3]:
sensor_df = pd.read_excel(data_path / "sensor_requirements.xlsx")
sensor_df = sensor_df[["requirement"]].copy()

actuator_df = pd.read_excel(data_path / "actuator_requirements.xlsx")

In [4]:
# no duplicate requirements
actuator_df.merge(sensor_df, how="inner", left_on="Requirement", right_on="requirement")[["Requirement"]].shape

(0, 1)

In [5]:
responses_df = pd.DataFrame(responses)

In [6]:
sensor_responses = sensor_df.merge(responses_df, how="left", left_on="requirement", right_on="requirement")[["requirement", "target_actuator","ai_answer"]]
sensor_responses.isna().sum()

requirement        0
target_actuator    0
ai_answer          0
dtype: int64

In [7]:
actuator_responses = actuator_df.merge(responses_df, how="left", left_on="Requirement", right_on="requirement")[["Requirement", "target_actuator","ai_answer"]]
actuator_responses.isna().sum()

Requirement        0
target_actuator    0
ai_answer          0
dtype: int64

In [8]:
actuator_responses.shape[0] + sensor_responses.shape[0]

182

In [9]:
responses_df.shape

(174, 10)

In [10]:
sensor_incorrect = sensor_responses.loc[sensor_responses["ai_answer"] != sensor_responses["target_actuator"]]
sensor_incorrect.sample(5)

,requirement,target_actuator,ai_answer
69,Yaw rate control systems must interact seamles...,0,1
93,The steering torque system must include an aut...,0,1
37,"In the event of oversteer, the vehicle dynamic...",0,1
47,The ESC system must utilize wheel speed data t...,0,1
24,The steering system must transfer the driver's...,0,1


In [11]:
actuator_incorrect = actuator_responses.loc[actuator_responses["ai_answer"] != actuator_responses["target_actuator"]]
actuator_incorrect

,Requirement,target_actuator,ai_answer
8,The electronic throttle control shall incorpor...,1,0
10,The vehicle system shall verify successful eng...,1,0
12,The engine control must transition into a limp...,1,0
33,All signal processes must be isolated with a n...,1,0
38,The system shall be designed so that single er...,1,0


In [12]:
sensor_df_full = pd.read_excel(data_path / "sensor_requirements.xlsx")
sensor_df_incorrect = sensor_df_full.merge(sensor_incorrect, how="inner", on="requirement")

In [13]:
sensor_df_incorrect["num_sensors"] = sensor_df_incorrect.drop(columns=["requirement", "target_actuator", "ai_answer"]).sum(axis=1)

In [14]:
sensor_df_incorrect["num_sensors"].value_counts()

num_sensors
1    64
2    16
Name: count, dtype: int64

In [16]:
sensor_df_incorrect.to_excel(data_path / "sensor_incorrect.xlsx", index=False)
actuator_incorrect.to_excel(data_path / "actuator_incorrect.xlsx", index=False)

In [29]:
sensor_df_correct = sensor_df_full.loc[~ sensor_df_full["requirement"].isin(sensor_df_incorrect["requirement"])]

In [30]:
sensor_df_correct.to_excel(data_path / "sensor_correct.xlsx", index=False)

### LLM rewrite the requirement

In [49]:
from openai import AzureOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

True

In [52]:
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_API_VERSION"),
)

In [53]:
# system prompt
messages = [
    {
        "role": "system",
        "content": f"""You are an expert in requirement engineering.
<Goal>
Rewrite the following requirement to be more like the style of the example following strictly
the format of the example and do not mess or lose any of the meaning nor the information.
---
<Examples>
{"\n".join(sensor_df_correct["requirement"].to_list())}
""",
    },
    {
        "role": "user",
        "content": f"""<Requirements>
{"\n".join(sensor_df_incorrect["requirement"].to_list())}
""",
    },
]


response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    temperature=0.2,
    seed=42,
)

In [60]:
new_reqs = response.choices[0].message.content.split(".  \n")

In [ ]:
new_reqs_df = pd.DataFrame(new_reqs, columns=["requirement"])

In [ ]:
sensor_df_incorrect["new_requirement"] = new_reqs_df["requirement"]
sensor_df_incorrect.to_excel(data_path / "sensor_reqs_rewritten.xlsx", index=False)